In [0]:
from pyspark.sql.functions import (
    to_date, lag, avg, stddev, max, min, lit,
    weekofyear, month, row_number, col, count, when, last, round, year
)
from pyspark.sql.types import DateType, StructType, StructField, StringType, DoubleType, LongType
from pyspark.sql.window import Window

In [0]:
storage_account_name = "stockmarket001"
container_name = "stock-data"
file_path = "raw/2026/03/22/"

required_columns = ["date", "sector", "open", "close", "low", "high"]

In [0]:
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net",
    access_key
)

In [0]:

schema = StructType([
    StructField("date", StringType(), True),
    StructField("sector", StringType(), True),
    StructField("open", DoubleType(), True),
    StructField("close", DoubleType(), True),
    StructField("low", DoubleType(), True),
    StructField("high", DoubleType(), True)
])

In [0]:
df = spark.read.csv(
    f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/{file_path}",
    header=True,
    schema=schema
)

In [0]:
# Select expected columns based on schema
expected_cols = [f.name for f in schema]
df = df.select(*expected_cols)

# Ensure all expected columns exist
for col_name in expected_cols:
    if col_name not in df.columns:
        df = df.withColumn(col_name, lit(None))

for c in df.columns:
    sanitized_name = c.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("₹", "inr").replace("-", "_")
    if sanitized_name != c:
        df = df.withColumnRenamed(c, sanitized_name)

df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

In [0]:
from pyspark.sql.functions import col, last, row_number, trim
from pyspark.sql.window import Window

def clean_and_deduplicate(df, date_col="date", order_col="close", partition_col="sector"):
    # Keep date as string and remove spaces
    df = df.withColumn(date_col, trim(col(date_col)))

    # Deduplicate per date+sector
    windowSpec = Window.partitionBy(date_col, partition_col).orderBy(col(order_col).desc())
    df = df.withColumn("row_num", row_number().over(windowSpec)) \
           .filter(col("row_num") == 1) \
           .drop("row_num")

    # Forward-fill per sector
    windowSpec_ffill = Window.partitionBy(partition_col).orderBy(date_col).rowsBetween(Window.unboundedPreceding, 0)
    df = df.withColumn("close", last("close", True).over(windowSpec_ffill)) \
           .withColumn("open", last("open", True).over(windowSpec_ffill)) \
           .withColumn("high", last("high", True).over(windowSpec_ffill)) \
           .withColumn("low", last("low", True).over(windowSpec_ffill)) \
           .dropna(subset=[date_col, partition_col, "close"])

    return df

df_clean = clean_and_deduplicate(df, date_col="date", order_col="close", partition_col="sector")


In [0]:
df_clean = clean_and_deduplicate(df, date_col="date", order_col="Close")
from pyspark.sql.functions import to_date
df_clean = df_clean.withColumn("date", to_date(col("date"), "yyyy-MM-dd"))

In [0]:
def add_moving_averages(df, column_name="close", sma_windows=[7], ema_windows=[7], partition_col=None):
    if partition_col:
        df = df.orderBy(partition_col, "date")
        base_window = Window.partitionBy(partition_col).orderBy("date")
    else:
        df = df.orderBy("date")
        base_window = Window.orderBy("date")

    for w in sma_windows:
        sma_window = base_window.rowsBetween(-w + 1, 0)
        df = df.withColumn(f"sma_{w}", avg(col(column_name)).over(sma_window))

    for w in ema_windows:
        alpha = 2 / (w + 1)
        df = df.withColumn(f"ema_{w}", (alpha * col(column_name)) + ((1 - alpha) * col(column_name)))

    return df

In [0]:
def add_ema(df, column_name="close", window_size=7, partition_col=None):
    alpha = 2 / (window_size + 1)

    if partition_col:
        windowSpec = Window.partitionBy(partition_col).orderBy("date")
    else:
        windowSpec = Window.orderBy("date")

    df = df.withColumn(f"ema_{window_size}", col(column_name))
    df = df.withColumn(f"ema_{window_size}", alpha * col(column_name) + (1 - alpha) * lag(f"ema_{window_size}", 1).over(windowSpec))
    return df

In [0]:
def apply_feature_engineering(df, partition_col=None):
    df = add_moving_averages(df, column_name="close", sma_windows=[7, 14], ema_windows=[], partition_col=partition_col)
    df = add_ema(df, "close", 7, partition_col)
    df = add_ema(df, "close", 14, partition_col)
    return df

In [0]:
df_features = apply_feature_engineering(df_clean)

In [0]:
def add_daily_return(df, price_col="close", output_col="daily_return", pct_col="daily_return_pct", partition_col=None):
    if partition_col:
        windowSpec = Window.partitionBy(partition_col).orderBy("date")
        df = df.orderBy(partition_col, "date")
    else:
        windowSpec = Window.orderBy("date")
        df = df.orderBy("date")

    prev_price = lag(price_col).over(windowSpec)

    df = df.withColumn(output_col, when(prev_price.isNull(), 0).when(prev_price == 0, 0).otherwise((col(price_col) - prev_price) / prev_price))
    df = df.withColumn(pct_col, col(output_col) * 100)

    return df

In [0]:
df = add_daily_return(df_clean, price_col="close")

In [0]:
def add_volatility(df, return_col="daily_return", windows=[7], partition_col=None):
    if partition_col:
        base_window = Window.partitionBy(partition_col).orderBy("date")
        df = df.orderBy(partition_col, "date")
    else:
        base_window = Window.orderBy("date")
        df = df.orderBy("date")

    for w in windows:
        rolling_window = base_window.rowsBetween(-w + 1, 0)
        df = df.withColumn(f"volatility_{w}", round(stddev(col(return_col)).over(rolling_window), 6))

    return df

In [0]:
df = add_volatility(df, return_col="daily_return", windows=[7, 14, 30])

In [0]:
def save_to_delta_table_managed(df, table_name, mode="overwrite"):
    df.write.format("delta").mode(mode).saveAsTable(table_name)
    print(f"Managed Delta table '{table_name}' created")

save_to_delta_table_managed(df, "sales_data")

In [0]:
def save_to_delta_table_managed(df, table_name, mode="overwrite"):
    df.write.format("delta").mode(mode).saveAsTable(table_name)
    print(f"Managed Delta table '{table_name}' created")

save_to_delta_table_managed(df, "sales_data")